Define env

In [0]:
import logging
from pyspark.sql import DataFrame, SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import col
from pyspark.sql.utils import AnalysisException
import os, sys
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
    
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")

        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration

        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    print("mode is shared")
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
catalog = f"`{env_prefix}-cdh`"

In [0]:
print(catalog)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated Zone')
dbutils.widgets.text('landed_zone', 'None', '02-Catalog Landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference Zone')
dbutils.widgets.text('curated_table', 'None', '04-Curated Table')
dbutils.widgets.text('run_date', 'None', '05-Run Date')
dbutils.widgets.text('schema', 'None', '06-Schema')
dbutils.widgets.text('externalfiles_zone', 'None', '07-Externalfiles Zone')
externalfiles_zone = dbutils.widgets.get('externalfiles_zone')
curated_zone = dbutils.widgets.get('curated_zone')
landed_zone = dbutils.widgets.get('landed_zone')
reference_zone = dbutils.widgets.get('reference_zone')
notebook = dbutils.widgets.get('curated_table')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')


In [0]:
relative_path_auth = "../helpers/authentication_helpers"
path = os.path.abspath(relative_path_auth)
if path not in sys.path:
    sys.path.append(path)
from authentication_utils import *


In [0]:
print(f"Curated Zone: {curated_zone}")

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
#env_prefix = landed_zone.split("_")[0]

curated_area = f"{catalog}.`{curated_zone}`"
landed_area = f"{catalog}.`{landed_zone}`"
reference_area = f"{catalog}.`{reference_zone}`"
externalfiles_area = f"{catalog}.`{externalfiles_zone}`"



In [0]:
print(curated_area)

In [0]:
path = "./curated_notebooks"
nb_path = f"{path}/{notebook}"
notebook_timeout = 7200

# Add pipeline_log_helpers to path
relative_path_log = "../helpers/lng_lnd"
log_path = os.path.abspath(relative_path_log)
if log_path not in sys.path:
    sys.path.append(log_path)

# DEBUG: try each run_id source independently (tags() may be blocked on shared clusters)
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
try:
    print('DEBUG currentRunId:', _ctx.currentRunId())
except Exception as _e:
    print('DEBUG currentRunId error:', _e)
try:
    print('DEBUG jobId:', _ctx.jobId())
except Exception as _e:
    print('DEBUG jobId error:', _e)
try:
    print('DEBUG idInJob:', _ctx.idInJob())
except Exception as _e:
    print('DEBUG idInJob error:', _e)
try:
    print('DEBUG rootRunId:', _ctx.rootRunId())
except Exception as _e:
    print('DEBUG rootRunId error:', _e)

# Log task start (run_id auto-detected inside log_task_start)
try:
    from pipeline_log_helpers import log_task_start, log_task_end
    log_task_start(spark, "curated", run_date, notebook, env_prefix=env_prefix)
    _log_enabled = True
except Exception as log_err:
    print(f"[WARNING] Task log start failed (non-blocking): {log_err}")
    _log_enabled = False

# Run curated notebook
try:
    dbutils.notebook.run(nb_path, notebook_timeout, {"curated_zone": curated_area, "catalog_landed": landed_area, "reference_zone": reference_area, "externalfiles_zone": externalfiles_area, "run_date": run_date, "schema": schema, "env_prefix": env_prefix})
    print(f" ...Done with table: {notebook}...")
    if _log_enabled:
        try:
            log_task_end(spark, "curated", run_date, notebook, status="completed", env_prefix=env_prefix)
        except Exception as log_err:
            print(f"[WARNING] Task log end failed (non-blocking): {log_err}")
except Exception as e:
    if _log_enabled:
        try:
            log_task_end(spark, "curated", run_date, notebook, status="failed", error_message=str(e)[:500], env_prefix=env_prefix)
        except Exception:
            pass
    raise
